# 01 — Language Detection
Phase 2 notebook: EDA, feature inspection, model evaluation deep-dive.

> Prerequisites: Phase 1 splits exist + `python scripts/01_train_language_detector.py` has been run.

In [ ]:
import sys, pickle
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SPLITS    = ROOT / 'data' / 'splits'
MODEL_DIR = ROOT / 'models' / 'language_detection'

plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})
print('Ready')

## 1. Exploratory Data Analysis

In [ ]:
train = pd.read_csv(SPLITS / 'language_train.csv')
val   = pd.read_csv(SPLITS / 'language_val.csv')
test  = pd.read_csv(SPLITS / 'language_test.csv')

print(f'train={len(train):,}  val={len(val):,}  test={len(test):,}')
train.head(4)

In [ ]:
# ── Language distribution ────────────────────────────────────────────────────
counts = train['language'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(13, 4))
counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Training Set — Language Distribution', fontsize=13)
ax.set_xlabel('Language code')
ax.set_ylabel('Samples')
ax.axhline(counts.mean(), color='red', linestyle='--', linewidth=1,
           label=f'mean = {counts.mean():.0f}')
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print('Class imbalance (max/min ratio):', f"{counts.max()/counts.min():.2f}x")

In [ ]:
# ── Text length distribution ─────────────────────────────────────────────────
train['word_count'] = train['text'].str.split().apply(len)
train['char_count'] = train['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train['word_count'], bins=40, color='teal', edgecolor='white')
axes[0].set_title('Word count distribution'); axes[0].set_xlabel('Words')
axes[1].hist(train['char_count'], bins=40, color='coral', edgecolor='white')
axes[1].set_title('Character count distribution'); axes[1].set_xlabel('Chars')
plt.tight_layout()
plt.show()
train[['word_count','char_count']].describe().round(1)

In [ ]:
# ── Sample texts from 5 languages ────────────────────────────────────────────
LANG_NAMES = {'en':'English','fr':'French','ar':'Arabic','ja':'Japanese','ru':'Russian'}
for lang, name in LANG_NAMES.items():
    sample = train[train['language']==lang]['text'].iloc[0]
    print(f"  [{lang}] {name}:")
    print(f"    {sample[:120]}")
    print()

## 2. Feature Inspection

In [ ]:
# Load saved vectorizer and model
with open(MODEL_DIR / 'vectorizer.pkl', 'rb') as f:
    vec = pickle.load(f)
with open(MODEL_DIR / 'model.pkl', 'rb') as f:
    clf = pickle.load(f)

print(f'Vocabulary size : {len(vec.vocabulary_):,}')
print(f'Classes         : {list(clf.classes_)}')

In [ ]:
# Top 10 char n-grams per language
feature_names = np.array(vec.get_feature_names_out())
SHOW = ['en', 'fr', 'de', 'ar', 'ja', 'zh']

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, lang in zip(axes.flat, SHOW):
    idx = list(clf.classes_).index(lang)
    top_idx = np.argsort(clf.coef_[idx])[-10:][::-1]
    top_feat = feature_names[top_idx]
    top_vals = clf.coef_[idx][top_idx]
    ax.barh(range(10), top_vals[::-1], color='steelblue')
    ax.set_yticks(range(10))
    ax.set_yticklabels([repr(f) for f in top_feat[::-1]], fontsize=8)
    ax.set_title(f'{lang} — top char n-grams', fontsize=10)
    ax.set_xlabel('LR coefficient')
plt.suptitle('Most discriminative char n-grams per language', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 3. Evaluation

In [ ]:
X_test = vec.transform(test['text'])
y_pred = clf.predict(X_test)
y_test = test['language']

acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {acc:.4f}  ({acc*100:.2f}%)')
print()
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            linewidths=0.3, ax=ax)
ax.set_title(f'Confusion Matrix  (acc={acc:.4f})', fontsize=13)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# Per-language F1 bar chart
report = classification_report(y_test, y_pred, output_dict=True)
langs = [l for l in labels if l in report]
f1s   = [report[l]['f1-score'] for l in langs]

fig, ax = plt.subplots(figsize=(13, 4))
colors = ['#e74c3c' if f < 0.90 else '#2ecc71' for f in f1s]
ax.bar(langs, f1s, color=colors, edgecolor='white')
ax.axhline(0.90, color='grey', linestyle='--', linewidth=1, label='0.90 target')
ax.axhline(0.95, color='orange', linestyle=':', linewidth=1, label='0.95 target')
ax.set_ylim(0.80, 1.02)
ax.set_title('Per-language F1 Score', fontsize=13)
ax.set_ylabel('F1')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Wrapper Class Demo

In [ ]:
from src.modules.language_detector import LanguageDetector

detector = LanguageDetector()

examples = [
    "I feel really anxious and can't sleep at night.",
    "Je me sens triste et incompris depuis quelques semaines.",
    "Tengo mucho miedo y no sé qué hacer con mi vida.",
    "私は最近とても気分が落ち込んでいます。",
    "أشعر بالقلق الشديد ولا أستطيع التركيز.",
    "Ich habe Angst und fühle mich sehr allein.",
]

print(f'{'Text':<50}  {'Lang':<6}  {'Conf':>6}')
print('-' * 65)
for text in examples:
    r = detector.detect(text)
    print(f"{text[:48]:<50}  {r['language']:<6}  {r['confidence']:>6.3f}")

In [ ]:
# Visualise confidence scores for one example
result = detector.detect("I feel really anxious and can't sleep at night.")
top10 = dict(list(result['all_scores'].items())[:10])

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(top10.keys(), top10.values(), color='steelblue', edgecolor='white')
ax.set_title("Confidence scores: 'I feel really anxious…'", fontsize=11)
ax.set_ylabel('Probability')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()